# Sweep functions

In [ ]:
def zigzag(x_inicio, x_fin, y_inicio, y_final, paso_x, paso_y, z):
    """
    Genera una lista de puntos (X, Y, Z) en un patrón de barrido tipo raster.
    El barrido se hace línea por línea (zigzag).
    """
    puntos = []
    y = y_inicio
    direccion = 1  # 1 = avanza en X positivo, -1 = X negativo

    while y <= y_final:
        if direccion == 1:
            xs = list(range(x_inicio, x_fin + paso_x, paso_x))
        else:
            xs = list(range(x_fin, x_inicio - paso_x, -paso_x))

        for x in xs:
            puntos.append((x, y, z))

        y += paso_y
        direccion *= -1  # invierte dirección en siguiente línea

    return puntos

In [ ]:
def recorrido_horizontal(x_inicio, x_fin, y_inicio, y_fin, paso_x, paso_y, z):
    puntos = []
    y = y_inicio

    while y <= y_fin:
        # Avanza en X de inicio a fin
        for x in range(x_inicio, x_fin + paso_x, paso_x):
            if x <= x_fin:
                puntos.append((x, y, z))
        
        # Avanza a la siguiente línea
        y += paso_y

        # Si aún no hemos terminado, subimos en Y y añadimos solo el punto de inicio
        if y <= y_fin:
            puntos.append((x_inicio, y, z))

    # Eliminar duplicados consecutivos (por seguridad)
    puntos_limpios = [puntos[0]]
    for p in puntos[1:]:
        if p != puntos_limpios[-1]:
            puntos_limpios.append(p)
            
    return puntos_limpios

In [ ]:
def generar_gcode(puntos, feedrate=500, tiempo_espera_1=0.5, tiempo_espera_2=5, z_seguro=-3, z_trabajo=0):
    """
    Genera un G-code que recorre una lista de puntos (X, Y, Z) realizando:
    - Movimiento a posición segura
    - Descenso a Z de trabajo
    - Activación/desactivación de sensor
    - Retorno a altura segura
    """
    gcode = []
    gcode.append("; --- Inicio del programa G-code ---")
    gcode.append("G21 ; Unidades en milimetros")
    gcode.append("G90 ; Coordenadas absolutas")
    gcode.append("G0 Z{:.3f} ; Bajar a altura segura inicial".format(z_seguro))
    gcode.append("")

    for i, (x, y, z) in enumerate(puntos):
        gcode.append(f"; --- Movimiento {i+1} a X{x} Y{y} ---")

        # 1. Mover sobre el punto a altura segura
        gcode.append(f"G0 X{x:.3f} Y{y:.3f} Z{z_seguro:.3f}")

        # 2. Bajar al plano de trabajo
        gcode.append(f"G1 Z{z_trabajo:.3f} F{feedrate}")

        # 3. Activar sensor
        gcode.append("M8 ; Activar Barrelino")
        gcode.append(f"G4 P{tiempo_espera_1} ; Esperar {tiempo_espera_1}s")

        # 4. Desactivar sensor
        gcode.append("M9 ; Desactivar Barrelino")

        # 5. Esperar antes del siguiente movimiento
        gcode.append(f"G4 P{tiempo_espera_2} ; Esperar {tiempo_espera_2}s")

        # 6. Subir a altura segura
        gcode.append(f"G0 Z{z_seguro:.3f}\n")


    gcode.append("; --- Final del recorrido ---")
    gcode.append("G0 X0 Y0 Z{:.3f} ; Regresar al origen seguro".format(z_seguro))
    gcode.append("M30 ; Fin del programa")
    gcode.append("; --- Fin del G-code ---")

    return "\n".join(gcode)

# Parameters

In [ ]:
x_inicio = 0
x_final = 40
paso_x = 5
y_inicio = 0
y_final = 30
paso_y = 5
z = 0

puntos = recorrido_horizontal(x_inicio, x_final, y_inicio, y_final, paso_x, paso_y, z)

gcode_texto = generar_gcode(puntos, feedrate=500, tiempo_espera_1=0.5, tiempo_espera_2=50, z_seguro=-3, z_trabajo=0)

with open("trayectoria.gcode", "w") as f:
    f.write(gcode_texto)

print("Generated G-code file")
